# Pré-processamento — Dataset CSEDM / ProgSnap2 v6

Transforma os dados brutos do CSEDM em sequências KT prontas para BKT, DKT e Code-DKT.  
Metodologia: fase de *Data Preparation* do EDM Process (Kalita et al., 2025).

---

## Splits do Dataset

O CSEDM disponibiliza dois semestres independentes, cada um com Train/Test próprios:

| Split | Semestre | Estudantes | Uso recomendado |
|-------|----------|------------|-----------------|
| `All/` | Fall-2019 (set–dez) | 506 | EDA exploratória completa |
| `Release/` | Spring-2019 (fev–mai) | 329 | **Comparação reproduzível com Shi et al. (2022)** |

As populações **não se sobrepõem**: SubjectIDs de `All/` ≠ SubjectIDs de `Release/`.  
Para replicar os resultados do paper Code-DKT (AUC ~74% em A1), usar sempre `Release/Train` para treino e `Release/Test` para teste.  
O benchmark de reprodutibilidade é **23.70% de tentativas corretas em Release/Train** (Shi et al. (2022) reporta 23.68% — margem de arredondamento esperada).

**Referência:** Shi et al. (2022); Pankiewicz, Shi & Baker (2025).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = ROOT / 'data' / 'CSEDM'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# Adicionar src/ ao path para importar data_loader
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_loader import load_main_table, load_labels

print(f'SEED = {SEED}')
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'DATA_ROOT existe: {DATA_ROOT.exists()}')

SEED = 42
DATA_ROOT = /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/data/CSEDM
DATA_ROOT existe: True


---

## 1 — Setup e Carregamento dos Dados

### 1.1 — Carregamento dos splits

**Contexto:** O primeiro passo de qualquer pipeline de pré-processamento é carregar os dados e confirmar que os splits têm as dimensões esperadas. Aqui carregamos os quatro splits relevantes: `All/Data` (EDA completa), `Release/Train` e `Release/Test` (treinamento e avaliação reproduzíveis conforme o protocolo de Shi et al. (2022)).  
**Hipótese:** `All/Data` deve conter ~360 mil eventos e 506 estudantes. `Release/Train` deve ter ~134 mil eventos com 23.70% de corretos em `Run.Program`. `Release/Test` deve ter ~32 mil eventos e 83 estudantes.  
**Referência:** Price et al. (2020); Shi et al. (2022).

In [2]:
# Carregar todos os splits
all_main        = load_main_table('all',           DATA_ROOT)
release_train   = load_main_table('release_train', DATA_ROOT)
release_test    = load_main_table('release_test',  DATA_ROOT)

# Labels (predição final do pipeline KT)
early_train = load_labels('release_train', DATA_ROOT, which='early')
late_train  = load_labels('release_train', DATA_ROOT, which='late')
early_test  = load_labels('release_test',  DATA_ROOT, which='early')
late_test   = load_labels('release_test',  DATA_ROOT, which='late')

splits = {
    'All/Data':        all_main,
    'Release/Train':   release_train,
    'Release/Test':    release_test,
}

print(f'{"Split":<20} {"Eventos":>10} {"Estudantes":>12} {"Assignments":>13}')
print('-' * 58)
for name, df in splits.items():
    n_events    = len(df)
    n_subjects  = df['SubjectID'].nunique()
    n_assign    = df['AssignmentID'].nunique()
    print(f'{name:<20} {n_events:>10,} {n_subjects:>12,} {n_assign:>13,}')

Split                   Eventos   Estudantes   Assignments
----------------------------------------------------------
All/Data                360,176          506             5
Release/Train           134,508          246             5
Release/Test             32,372           83             3


**Achado:** Os splits têm as dimensões esperadas — `All/Data` com ~360k eventos e 506 estudantes; `Release/Train` com ~134k e 246 estudantes; `Release/Test` com ~32k e 83 estudantes. Populações sem sobreposição (semestres distintos).  
**Implicação para modelagem:** Toda comparação com os resultados de Shi et al. (2022) deve usar exclusivamente `Release/Train` e `Release/Test`. O split `All/` é reservado para análises exploratórias e não deve ser misturado com `Release/`.

### 1.2 — Benchmark de reprodutibilidade

**Contexto:** Shi et al. (2022) reporta que o dataset de treinamento contém 23.68% de tentativas corretas (Run.Program com Score == 1.0). Confirmar este valor é o primeiro teste de reprodutibilidade do pipeline.  
**Hipótese:** A taxa calculada em `Release/Train` deve ser ~23.68–23.70%, refletindo apenas diferença de arredondamento entre os papers.  
**Referência:** Shi et al. (2022) — Section 4.1, "23.68% of the attempts are correct".

In [3]:
runs_train = release_train[release_train['EventType'] == 'Run.Program'].copy()
runs_test  = release_test[release_test['EventType']  == 'Run.Program'].copy()

correct_rate_train = (runs_train['Score'] == 1.0).mean()
correct_rate_test  = (runs_test['Score']  == 1.0).mean()

print(f'Release/Train — Run.Program: {len(runs_train):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_train:.4%}')
print(f'  Paper Shi et al. (2022):          23.6800%')
print(f'  Divergência:                      {abs(correct_rate_train - 0.2368):.4%}')
print()
print(f'Release/Test — Run.Program: {len(runs_test):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_test:.4%}')

Release/Train — Run.Program: 46,825 eventos
  Taxa de corretos (Score == 1.0): 23.7010%
  Paper Shi et al. (2022):          23.6800%
  Divergência:                      0.0210%

Release/Test — Run.Program: 10,845 eventos
  Taxa de corretos (Score == 1.0): 21.1803%


**Achado:** A taxa de corretos em `Release/Train` é ~23.70%, a menos de 0.02pp do valor de 23.68% reportado por Shi et al. (2022) — diferença de arredondamento esperada. O benchmark de reprodutibilidade é confirmado.  
**Implicação para modelagem:** O dataset está íntegro para reprodução do paper. A baixa proporção de acertos (~23.7%) justifica o uso de AUC como métrica primária em vez de acurácia — uma predição trivial "sempre errado" teria acurácia de 76.3%, mas AUC de ~50%.

### 1.3 — Estrutura e tipos de dados

**Contexto:** Verificar se as colunas críticas estão presentes e com tipos corretos. Os modelos KT dependem de `SubjectID`, `AssignmentID`, `ProblemID`, `ServerTimestamp` (ordem cronológica), `EventType` e `Score`.  
**Hipótese:** Todas as colunas críticas devem estar presentes em `Release/Train`. O `Release/` tem uma coluna extra `CourseSectionID` e `SourceLocation` ausentes em `All/`.  
**Referência:** Price et al. (2020) — ProgSnap2 v6 specification.

In [4]:
CRITICAL_COLS = ['SubjectID', 'AssignmentID', 'ProblemID', 'ServerTimestamp',
                 'EventType', 'Score', 'CodeStateID']

print('=== Release/Train — dtypes e nulos ===')
info = pd.DataFrame({
    'dtype':  release_train.dtypes.astype(str),
    'nulls':  release_train.isna().sum(),
    'null_%': (release_train.isna().mean() * 100).round(2),
})
print(info.to_string())
print()

missing = [c for c in CRITICAL_COLS if c not in release_train.columns]
if missing:
    print(f'AVISO — colunas críticas ausentes: {missing}')
else:
    print('OK — todas as colunas críticas presentes em Release/Train')

# Score só existe em Run.Program — nulo em Compile e Compile.Error é esperado
score_null_outside_run = release_train[
    (release_train['EventType'] != 'Run.Program') & release_train['Score'].notna()
]
print(f'Score não-nulo fora de Run.Program: {len(score_null_outside_run)} (esperado: 0)')

=== Release/Train — dtypes e nulos ===
                                         dtype  nulls  null_%
Order                                    int64      0    0.00
SubjectID                                  str      0    0.00
ToolInstances                              str      0    0.00
ServerTimestamp            datetime64[us, UTC]      0    0.00
ServerTimezone                             str      0    0.00
CourseID                                   str      0    0.00
CourseSectionID                          int64      0    0.00
AssignmentID                             Int64      0    0.00
ProblemID                                Int64      0    0.00
CodeStateID                                str      0    0.00
IsEventOrderingConsistent                 bool      0    0.00
EventType                                  str      0    0.00
Score                                  float64  87683   65.19
Compile.Result                             str  87683   65.19
CompileMessageType             

**Achado:** Todas as colunas críticas estão presentes. `Score` é nulo em `Compile` e `Compile.Error` (comportamento esperado — Score só existe em `Run.Program`). `ServerTimestamp` foi convertido para `datetime64[ns, UTC]` pelo `load_main_table`, garantindo ordenação cronológica correta.  
**Implicação para modelagem:** Nenhuma engenharia de colunas adicional é necessária para o carregamento. O `load_main_table` de `src/data_loader.py` entrega o DataFrame pronto para as etapas de filtragem (tarefa 2).